# Gold Layer: Feature Engineering for ML Classifier

This notebook extracts features from the annotated gold layer for training a
spaceflight vs ground control classifier.

**Input:** `data/gold/osd352_brain_v1_annotated.h5ad`

**Output:** `data/gold/osd352_brain_v1_features.parquet`

---

### Why pseudobulk per sample × cell type?

We have 27,968 cells but only 5 biological samples (3 Flight, 2 GC).
Using individual cells as ML samples would be **pseudoreplication** — cells from
the same mouse are not independent observations. A classifier trained this way
would learn mouse-specific patterns, not spaceflight effects, and would fail
on new data.

Instead, we aggregate (pseudobulk) by **sample × cell type**:
- 5 samples × ~11 cell types = ~55 pseudobulk profiles
- Each profile is a biologically meaningful unit
- Avoids inflating sample size with non-independent observations
- Standard approach in modern scRNA-seq differential analysis

Features per pseudobulk profile:
1. Mean expression of top DE genes
2. Cell type proportions per sample
3. QC summary metrics (mean genes, UMI, mt%)

## Step 1: Load Gold Layer

In [1]:
# Standard library
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import scanpy as sc

# Load gold layer
adata = sc.read_h5ad('../data/gold/osd352_brain_v1_annotated.h5ad')

print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"Samples: {adata.obs['sample_name'].nunique()}")
print(f"Cell types: {adata.obs['cell_type'].nunique()}")
print(f"Conditions: {adata.obs['condition'].value_counts().to_dict()}")

Loaded: 27968 cells × 32285 genes
Samples: 5
Cell types: 67
Conditions: {'Space Flight': 17972, 'Ground Control': 9996}


## Step 2: Load DE Results and Select Top Genes

Use the top differentially expressed genes as features.
These are the genes most likely to distinguish spaceflight from ground control.

In [2]:
# Load DE results
de_all = pd.read_parquet('../data/gold/osd352_brain_v1_de_results.parquet')

# Select significant genes with meaningful fold change
sig_genes = de_all[
    (de_all['pvals_adj'] < 0.05) &
    (de_all['logfoldchanges'].abs() > 0.5)
].copy()

# Get unique gene names (a gene can appear in multiple cell types)
top_gene_names = sig_genes['names'].unique()

# Filter to genes present in adata
top_gene_names = [g for g in top_gene_names if g in adata.var_names]

print(f"Significant DE genes (adj p < 0.05, |logFC| > 0.5): {len(sig_genes)}")
print(f"Unique genes for features: {len(top_gene_names)}")

Significant DE genes (adj p < 0.05, |logFC| > 0.5): 4297
Unique genes for features: 3240


## Step 3: Pseudobulk Aggregation

For each sample × cell type combination:
- Compute mean expression of top DE genes
- This creates one row per pseudobulk profile

In [3]:
# Get the cell types that were tested in DE (>= 50 cells per condition)
valid_cell_types = adata.uns['de_spaceflight_vs_ground']['cell_types_tested']
print(f"Cell types for pseudobulk: {len(valid_cell_types)}")

# Build pseudobulk expression matrix
pseudobulk_rows = []

for sample in adata.obs['sample_name'].unique():
    for cell_type in valid_cell_types:
        # Subset to this sample + cell type
        mask = (adata.obs['sample_name'] == sample) & (adata.obs['cell_type'] == cell_type)
        subset = adata[mask]

        if subset.n_obs < 5:  # skip if too few cells
            continue

        # Mean expression of top DE genes
        # Use raw counts for more robust aggregation
        expr_data = subset[:, top_gene_names].X
        if hasattr(expr_data, 'toarray'):
            expr_data = expr_data.toarray()
        mean_expr = np.mean(expr_data, axis=0)

        # Build row
        row = {
            'sample_name': sample,
            'cell_type': cell_type,
            'condition': subset.obs['condition'].iloc[0],
            'n_cells': subset.n_obs
        }

        # Add gene expression features
        for i, gene in enumerate(top_gene_names):
            row[f'expr_{gene}'] = mean_expr[i]

        pseudobulk_rows.append(row)

pseudobulk_df = pd.DataFrame(pseudobulk_rows)
print(f"\nPseudobulk profiles: {len(pseudobulk_df)}")
print(f"Features (genes): {len(top_gene_names)}")
print(f"\nProfiles per condition:")
print(pseudobulk_df['condition'].value_counts())

Cell types for pseudobulk: 11

Pseudobulk profiles: 54
Features (genes): 3240

Profiles per condition:
condition
Space Flight      32
Ground Control    22
Name: count, dtype: int64


## Step 4: Cell Type Proportion Features

The fraction of each cell type within a sample is itself a feature.
Spaceflight changes cell composition (e.g., 4.4x more microglia) —
these proportions carry biological signal.

In [4]:
# Compute cell type proportions per sample
prop_rows = []

for sample in adata.obs['sample_name'].unique():
    sample_mask = adata.obs['sample_name'] == sample
    total_cells = sample_mask.sum()
    condition = adata.obs.loc[sample_mask, 'condition'].iloc[0]

    row = {'sample_name': sample, 'condition': condition}

    for cell_type in valid_cell_types:
        ct_mask = sample_mask & (adata.obs['cell_type'] == cell_type)
        ct_count = ct_mask.sum()
        proportion = ct_count / total_cells
        row[f'prop_{cell_type}'] = proportion

    prop_rows.append(row)

prop_df = pd.DataFrame(prop_rows)
print(f"Proportion features: {len(valid_cell_types)} cell types × {len(prop_df)} samples")
print(prop_df.set_index('sample_name').drop(columns='condition').round(4))

Proportion features: 11 cell types × 5 samples
                prop_181 IC Tfap2d Maf Glut  \
sample_name                                   
RR3_BRN_FLT_F1                       0.0248   
RR3_BRN_GC_G9                        0.0076   
RR3_BRN_GC_G8                        0.0125   
RR3_BRN_FLT_F2                       0.0053   
RR3_BRN_FLT_F7                       0.0114   

                prop_243 PGRN-PARN-MDRN Hoxb5 Glut  \
sample_name                                          
RR3_BRN_FLT_F1                              0.0286   
RR3_BRN_GC_G9                               0.0218   
RR3_BRN_GC_G8                               0.0211   
RR3_BRN_FLT_F2                              0.0214   
RR3_BRN_FLT_F7                              0.0137   

                prop_311 CBX MLI Megf11 Gaba  prop_312 CBX MLI Cdh22 Gaba  \
sample_name                                                                 
RR3_BRN_FLT_F1                        0.0238                       0.0051   
RR3_BRN_GC_G9

## Step 5: QC Summary Features

Per-sample QC metrics as additional features.
Spaceflight affects overall transcriptional activity and cell health.

In [5]:
# QC features per sample × cell type
qc_features = []

for _, row in pseudobulk_df[['sample_name', 'cell_type']].iterrows():
    mask = (
        (adata.obs['sample_name'] == row['sample_name']) &
        (adata.obs['cell_type'] == row['cell_type'])
    )
    subset = adata.obs[mask]

    qc_features.append({
        'sample_name': row['sample_name'],
        'cell_type': row['cell_type'],
        'qc_mean_genes': subset['n_genes_by_counts'].mean(),
        'qc_mean_counts': subset['total_counts'].mean(),
        'qc_mean_mt_pct': subset['pct_counts_mt'].mean(),
        'qc_median_genes': subset['n_genes_by_counts'].median(),
        'qc_median_counts': subset['total_counts'].median()
    })

qc_df = pd.DataFrame(qc_features)
print(f"QC features: {qc_df.shape}")
print(qc_df.head())

QC features: (54, 7)
      sample_name                      cell_type  qc_mean_genes  \
0  RR3_BRN_FLT_F1         181 IC Tfap2d Maf Glut    1844.668966   
1  RR3_BRN_FLT_F1  243 PGRN-PARN-MDRN Hoxb5 Glut    2155.473054   
2  RR3_BRN_FLT_F1        311 CBX MLI Megf11 Gaba     970.230216   
3  RR3_BRN_FLT_F1         312 CBX MLI Cdh22 Gaba     906.166667   
4  RR3_BRN_FLT_F1            314 CB Granule Glut     532.034471   

   qc_mean_counts  qc_mean_mt_pct  qc_median_genes  qc_median_counts  
0     3896.224121        0.138670           1730.0            3478.5  
1     4733.383301        0.137709           2042.5            4080.5  
2     1715.906494        0.248957            946.0            1620.0  
3     1517.316650        0.195191            934.5            1535.5  
4      817.311646        0.129510            472.0             691.0  


## Step 6: Merge All Features

Combine pseudobulk expression, cell type proportions, and QC metrics
into a single feature matrix.

In [6]:
# Merge pseudobulk expression with QC features
features_df = pseudobulk_df.merge(
    qc_df,
    on=['sample_name', 'cell_type'],
    how='left'
)

# Merge cell type proportions (sample-level, broadcast to all cell types)
features_df = features_df.merge(
    prop_df,
    on=['sample_name', 'condition'],
    how='left'
)

# Add binary label for ML
features_df['label'] = (features_df['condition'] == 'Space Flight').astype(int)

# Count feature columns (exclude metadata)
metadata_cols = ['sample_name', 'cell_type', 'condition', 'n_cells', 'label']
feature_cols = [c for c in features_df.columns if c not in metadata_cols]

print(f"Final feature matrix: {features_df.shape}")
print(f"Samples (rows): {len(features_df)}")
print(f"Features (columns): {len(feature_cols)}")
print(f"Metadata columns: {metadata_cols}")
print(f"\nLabel distribution:")
print(features_df['label'].value_counts().rename({0: 'Ground Control', 1: 'Space Flight'}))

Final feature matrix: (54, 3261)
Samples (rows): 54
Features (columns): 3256
Metadata columns: ['sample_name', 'cell_type', 'condition', 'n_cells', 'label']

Label distribution:
label
Space Flight      32
Ground Control    22
Name: count, dtype: int64


## Step 7: Save Feature Matrix

Save as Parquet for efficient loading in the ML training notebook.
This is the bridge between gold layer biology and the model layer.

In [7]:
# Save feature matrix
output_path = Path('../data/gold/osd352_brain_v1_features.parquet')
features_df.to_parquet(output_path, index=False)

file_size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"Saved: {output_path} ({file_size_mb:.2f} MB)")
print(f"Shape: {features_df.shape}")
print(f"\nReady for ML training in 06_model_training.ipynb")

Saved: ../data/gold/osd352_brain_v1_features.parquet (2.48 MB)
Shape: (54, 3261)

Ready for ML training in 06_model_training.ipynb


## Summary

**Feature engineering complete for OSD-352 brain tissue**

| Component | Details |
|-----------|--------|
| Aggregation | Pseudobulk per sample × cell type |
| Pseudobulk profiles | 54 (32 Space Flight, 22 Ground Control) |
| Expression features | 3,240 genes (significant DE: adj p < 0.05, \|logFC\| > 0.5) |
| Proportion features | 11 cell type fractions per sample |
| QC features | 5 metrics (mean/median genes, UMI counts, mt%) |
| Total features | 3,256 |
| Output | `data/gold/osd352_brain_v1_features.parquet` (2.5 MB) |

### Why pseudobulk per sample × cell type?
- **Avoids pseudoreplication**: 27,968 cells come from only 5 mice — cells from the same mouse are not independent
- **Biologically meaningful**: Each profile represents a specific cell type in a specific animal
- **Standard practice**: This is the accepted approach in modern scRNA-seq ML studies

### Feature composition
- **DE gene expression** (3,240 features): Mean expression of significantly differentially expressed genes per pseudobulk profile. These are biologically motivated — they distinguish spaceflight from ground control at the transcriptional level.
- **Cell type proportions** (11 features): Fraction of each cell type per sample. Captures compositional shifts (e.g., 4.4x microglia enrichment in spaceflight).
- **QC metrics** (5 features): Mean/median genes detected, UMI counts, and mitochondrial %. Captures overall transcriptional activity and cell health differences.

### Key observations from proportions
- CB Granule Glut: 69-73% in GC vs 24-54% in Flight (dominant population shrinks)
- Microglia: 0.7-0.8% in GC vs 1.4-2.7% in Flight (neuroinflammation signal)
- Oligo NN: Sample F2 has 16% vs 6-10% in others (sample-level variation)

**Next:** Model training with MLflow tracking (`06_model_training.ipynb`)